In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

from config.config import Config
from trainer import registry
from utils import utils

%cd ../src
%pwd


c:\Users\xunathan\workspace\deepkernel\src


c:\Users\xunathan\workspace\deepkernel\venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'c:\\Users\\xunathan\\workspace\\deepkernel\\src'

In [4]:
MODEL = 'mlp15x30x45x30-squared'
DATASET = 'hdgm30'
DEVICE = 'cuda:0'

CONFIG = f'config/exp/eval/deepkernel/hsic/hdgm/hsic.{MODEL}.{DATASET}.yml'
PRETRAINED_PATH = f'exp/pretrained/deepkernel/hsic/hdgm/{MODEL}/best/best.pt'

In [5]:
cfg = Config(yaml_path=CONFIG, device=DEVICE)
utils.seed_all(cfg['seed'])
pipeline = registry.get('HSIC').build(cfg)
pipeline.load_checkpoint(PRETRAINED_PATH)

k = pipeline.model['k']
l = pipeline.model['l']

print(k.eps)
print(l.eps)


tensor([0.0290], device='cuda:0', grad_fn=<SigmoidBackward0>)
tensor([0.0099], device='cuda:0', grad_fn=<SigmoidBackward0>)


In [12]:
settings = [
    ('hdgm4', 'mlp2x4x6x4-squared'),
    ('hdgm8', 'mlp4x8x12x8-squared'),
    ('hdgm10', 'mlp5x10x15x10-squared'),
    ('hdgm20', 'mlp10x20x30x20-squared'),
    ('hdgm30', 'mlp15x30x45x30-squared'),
    ('hdgm40', 'mlp20x40x60x40-squared'),
    ('hdgm50', 'mlp25x50x75x50-squared'),
]

param_dict = dict()

for dataset, model in settings:
    CONFIG = f'config/exp/eval/deepkernel/hsic/hdgm/hsic.{model}.{dataset}.yml'
    PRETRAINED_PATH = f'exp/pretrained/deepkernel/hsic/hdgm/{model}/best/best.pt'

    cfg = Config(yaml_path=CONFIG, device=DEVICE)
    utils.seed_all(cfg['seed'])
    pipeline = registry.get('HSIC').build(cfg)
    pipeline.load_checkpoint(PRETRAINED_PATH)

    k = pipeline.model['k']
    l = pipeline.model['l']

    param_dict[model] = {
        'k-eps': k.eps.item(),
        'l-eps': l.eps.item(),
    }

print(param_dict)


{'mlp2x4x6x4-squared': {'k-eps': 0.004431288223713636, 'l-eps': 0.004912798758596182}, 'mlp4x8x12x8-squared': {'k-eps': 0.021022463217377663, 'l-eps': 0.009190449491143227}, 'mlp5x10x15x10-squared': {'k-eps': 0.01136076170951128, 'l-eps': 0.013090692460536957}, 'mlp10x20x30x20-squared': {'k-eps': 0.011864307336509228, 'l-eps': 0.006379482802003622}, 'mlp15x30x45x30-squared': {'k-eps': 0.02903870679438114, 'l-eps': 0.00986531749367714}, 'mlp20x40x60x40-squared': {'k-eps': 0.015215241350233555, 'l-eps': 0.0073713394813239574}, 'mlp25x50x75x50-squared': {'k-eps': 0.005228782072663307, 'l-eps': 0.005331062246114016}}


In [14]:
eps = []
for model, params in param_dict.items():
    eps.append(params['k-eps'])
    eps.append(params['l-eps'])

mean = sum(eps) / len(eps)
var = sum([(x - mean)**2 for x in eps]) / len(eps)
std = var**0.5

print('max:', max(eps))
print('min:', min(eps))
print('mean:', mean)
print('std:', std)


max: 0.02903870679438114
min: 0.004431288223713636
mean: 0.01102162095984178
std: 0.006727230043261762
